# AutoGluon GPU High-Capacity Pipeline (Kaggle Tabular s6e10)
### Multi-Layer Stacking + PyTorch Tabular Neural Nets & Gradient Boosted Trees

This notebook uses the high-tier GPU environment to run AutoGluon on the **63 Sovereign Static Features**:
1. **Fixed 10-Fold CV**: Zero leakage; folds match the exact cross-validation split used by V19.
2. **Multi-Model Heterogeneous Diversity**: Trains **PyTorch Tabular Neural Networks (`NN_TORCH`)**, **FastAI**, **CatBoost (`CAT`)**, **LightGBM (`GBM`)**, and **XGBoost (`XGB`)**.
3. **Multi-Layer Stacking**: Stack level meta-learners that dynamically condition their ensemble weights.
4. **Out-of-Fold (OOF) Probability Export**: Saves clean OOF probabilities for post-hoc consensus ensembling with V19.

## 1. Environment & AutoGluon GPU Installation

In [ ]:
# Install AutoGluon with tabular and GPU acceleration in Colab if not present
!pip install -q --upgrade "autogluon.tabular[all]" torch torchvision torchaudio

import os, sys, glob, gc, time, warnings, random
import numpy as np
import pandas as pd
import torch
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold
from autogluon.tabular import TabularPredictor

warnings.filterwarnings('ignore')
SEED = 42

print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)}")

## 2. Load Data & Generate 63 Sovereign Features

In [ ]:
def find_file(names):
    for name in names:
        matches = glob.glob(name)
        if matches:
            return matches[0]
    return None

train_path = find_file(['data/raw/train.csv', 'data/train.csv', 'train.csv', '/kaggle/input/*/train.csv'])
test_path = find_file(['data/raw/test.csv', 'data/test.csv', 'test.csv', '/kaggle/input/*/test.csv'])

print(f"Train path: {train_path}")
print(f"Test path:  {test_path}")

train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)

TARGET = 'satisfaction'
if train_df[TARGET].dtype == object:
    train_df[TARGET] = (train_df[TARGET].astype(str).str.lower().str.strip() == 'satisfied').astype(int)

# Feature engineering - 63 static features
FD = 'Flight Distance'
CATS = ['Gender', 'Customer Type', 'Type of Travel', 'Class']
NUMS = ['Age', FD, 'Departure Delay in Minutes', 'Arrival Delay in Minutes']
RATINGS = [
    'Inflight wifi service', 'Departure/Arrival time convenient', 'Ease of Online booking',
    'Gate location', 'Food and drink', 'Online boarding', 'Seat comfort',
    'Inflight entertainment', 'On-board service', 'Leg room service',
    'Baggage handling', 'Checkin service', 'Cleanliness'
]
CORE_FEATS = NUMS + RATINGS + CATS

ntr = len(train_df)
full_df = pd.concat([train_df.drop(columns=[TARGET, 'id'], errors='ignore'),
                     test_df.drop(columns=['id'], errors='ignore')], ignore_index=True)

full_df['Arrival Delay in Minutes'] = full_df['Arrival Delay in Minutes'].fillna(full_df['Departure Delay in Minutes']).fillna(0.0)
full_df['Delay_Total'] = (full_df['Departure Delay in Minutes'] + full_df['Arrival Delay in Minutes']).astype(np.float32)
full_df['Delay_Diff'] = (full_df['Arrival Delay in Minutes'] - full_df['Departure Delay in Minutes']).astype(np.float32)
full_df['Delay_Ratio'] = ((full_df['Arrival Delay in Minutes'] + 1.0) / (full_df['Departure Delay in Minutes'] + 1.0)).astype(np.float32)
full_df['Delay_Per_Mile'] = (full_df['Delay_Total'] / (full_df[FD] + 1.0)).astype(np.float32)
full_df['Zero_Ratings_Count'] = (full_df[RATINGS] == 0).sum(axis=1).astype(np.float32)

full_df['Digital_Service_Index'] = ((full_df['Inflight wifi service'] + 1.0) * (full_df['Online boarding'] + 1.0)).astype(np.float32)
full_df['Digital_Satisfaction_Gap'] = (full_df['Inflight wifi service'] - full_df['Online boarding']).astype(np.float32)
full_df['Physical_Service_Index'] = ((full_df['Seat comfort'] + full_df['Cleanliness'] + full_df['Food and drink']) / 3.0).astype(np.float32)
full_df['Cabin_Comfort_Cluster'] = ((full_df['Seat comfort'] + full_df['Leg room service'] + full_df['Cleanliness']) / 3.0).astype(np.float32)
class_map = {'Eco': 0, 'Eco Plus': 1, 'Business': 2}
class_numeric = full_df['Class'].map(class_map).fillna(0).astype(np.float32)
full_df['Service_Expectation_Gap'] = (full_df['Physical_Service_Index'] - (class_numeric * 1.5)).astype(np.float32)
full_df['Wifi_Boarding_Ratio'] = ((full_df['Inflight wifi service'] + 0.5) / (full_df['Online boarding'] + 0.5)).astype(np.float32)
loyalty_num = (full_df['Customer Type'] == 'Loyal Customer').astype(np.float32)
biz_num = (full_df['Type of Travel'] == 'Business travel').astype(np.float32)
full_df['Loyal_Business_Segment'] = (loyalty_num * 2.0 + biz_num).astype(np.float32)

cnt_feats = pd.DataFrame(index=full_df.index)
for col in NUMS:
    cnt_feats[f'cnt_{col}'] = full_df[col].map(train_df[col].value_counts()).fillna(0).astype(np.float32)

coded_full = full_df[CORE_FEATS].copy()
for c in CATS:
    coded_full[c] = coded_full[c].astype('category').cat.codes
route_group = coded_full.groupby(FD)
route_profiles = pd.DataFrame(index=full_df.index)
for c in CORE_FEATS:
    if c != FD:
        route_profiles[f'route_mean_{c}'] = route_group[c].transform('mean').astype(np.float32)
route_profiles['route_size'] = route_group['Age'].transform('size').astype(np.float32)

orig_priors = pd.DataFrame(index=full_df.index)
orig_priors['orig_route_satisfaction_prior'] = 0.5
orig_priors['og_xgb_prob'] = 0.5
orig_priors['og_xgb_logit'] = 0.0
orig_priors['og_lgb_prob'] = 0.5
orig_priors['og_lgb_logit'] = 0.0

core_encoded = full_df[CORE_FEATS].copy()
for c in CATS:
    core_encoded[c] = core_encoded[c].astype('category')

context_cols = [
    'Delay_Total', 'Delay_Diff', 'Delay_Ratio', 'Delay_Per_Mile',
    'Zero_Ratings_Count', 'Digital_Service_Index', 'Digital_Satisfaction_Gap',
    'Physical_Service_Index', 'Cabin_Comfort_Cluster', 'Service_Expectation_Gap',
    'Wifi_Boarding_Ratio', 'Loyal_Business_Segment'
]

static_x = pd.concat([core_encoded, full_df[context_cols], cnt_feats, route_profiles, orig_priors], axis=1)

# Construct final training dataframe for AutoGluon
ag_train = static_x.iloc[:ntr].copy().reset_index(drop=True)
ag_train[TARGET] = train_df[TARGET].values
ag_test = static_x.iloc[ntr:].copy().reset_index(drop=True)

# Generate and attach identical 10-fold CV identifiers
skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=SEED)
ag_train['fold_id'] = 0
for f_idx, (_, v_idx) in enumerate(skf.split(ag_train, ag_train[TARGET])):
    ag_train.loc[v_idx, 'fold_id'] = f_idx

print(f"AutoGluon Train Shape: {ag_train.shape} (Features={ag_train.shape[1] - 2})")
print(f"AutoGluon Test Shape:  {ag_test.shape}")

## 3. Train AutoGluon Tabular Multi-Layer Ensembles
Configures multi-bagging and multi-layer stacking with PyTorch Tabular Neural Nets, FastAI, CatBoost, LightGBM, and XGBoost.

In [ ]:
# Multi-model hyperparameters targeting high-end GPU
custom_hyperparameters = {
    'NN_TORCH': {},
    'FASTAI': {},
    'GBM': [{'extra_trees': True, 'ag_args': {'name_suffix': 'XT'}}, {}],
    'CAT': {},
    'XGB': {}
}

os.makedirs('ag_models', exist_ok=True)
predictor = TabularPredictor(
    label=TARGET,
    eval_metric='roc_auc',
    path='ag_models'
)

predictor.fit(
    train_data=ag_train,
    presets='best_quality',
    hyperparameters=custom_hyperparameters,
    groups='fold_id',
    num_bag_folds=10,
    num_stack_levels=1,
    time_limit=3600 * 3,  # Adjust budget as needed (e.g. 3 hours)
    auto_stack=True
)

print("Training Complete!")
lb = predictor.leaderboard(silent=False)
display(lb)

## 4. Export AutoGluon OOF & Test Predictions

In [ ]:
os.makedirs('oof', exist_ok=True)
os.makedirs('submissions', exist_ok=True)

# 1. Out-of-fold probability predictions
oof_probs = predictor.predict_proba_oof()  # Returns pd.DataFrame or Series
if isinstance(oof_probs, pd.DataFrame):
    ag_oof = oof_probs[1].values
else:
    ag_oof = oof_probs.values

oof_auc = roc_auc_score(train_df[TARGET], ag_oof)
print(f"AutoGluon Stacker 10-Fold OOF ROC-AUC: {oof_auc:.6f}")

np.save('oof/autogluon_oof.npy', ag_oof)

# 2. Test probability predictions
test_probs = predictor.predict_proba(ag_test)
if isinstance(test_probs, pd.DataFrame):
    ag_test_preds = test_probs[1].values
else:
    ag_test_preds = test_probs.values

np.save('oof/autogluon_test.npy', ag_test_preds)

test_ids = test_df['id'].values if 'id' in test_df.columns else test_df.index
sub_ag = pd.DataFrame({'id': test_ids, TARGET: ag_test_preds})
sub_ag.to_csv('submissions/submission_autogluon.csv', index=False)
print("Exported AutoGluon predictions to submissions/submission_autogluon.csv")

## 5. Post-Hoc Consensus Blending with V19 Solution
Combines AutoGluon multi-layer neural/GBDT stacker predictions with V19 consensus trees using Rank Averaging.

In [ ]:
from scipy.stats import rankdata

v19_sub_path = 'submissions/submission.csv'  # V19 primary consensus submission
if os.path.exists(v19_sub_path):
    v19_sub = pd.read_csv(v19_sub_path)
    v19_preds = v19_sub[TARGET].values
    
    # Rank-averaged consensus blend (50% AutoGluon + 50% V19)
    rank_ag = rankdata(ag_test_preds) / len(ag_test_preds)
    rank_v19 = rankdata(v19_preds) / len(v19_preds)
    
    final_blend_probs = 0.50 * ag_test_preds + 0.50 * v19_preds
    final_blend_rank = 0.50 * rank_ag + 0.50 * rank_v19
    
    pd.DataFrame({'id': test_ids, TARGET: final_blend_probs}).to_csv('submissions/submission_ag_v19_consensus.csv', index=False)
    pd.DataFrame({'id': test_ids, TARGET: final_blend_rank}).to_csv('submissions/submission_ag_v19_rank_consensus.csv', index=False)
    print("Generated hybrid consensus submissions:")
    print(" * submissions/submission_ag_v19_consensus.csv")
    print(" * submissions/submission_ag_v19_rank_consensus.csv")
else:
    print("Notice: V19 submission.csv not found yet. Run airline_satisfaction_pipeline.ipynb to enable hybrid blending.")

In [ ]:
# ==============================================================================
# AUTOMATIC DOWNLOADING & GOOGLE DRIVE BACKUP ON COMPLETION
# ==============================================================================
import os, glob, shutil
print('[*] Pipeline complete! Preparing automatic download and Drive backup...')

# 1. If Google Drive is mounted, copy all submissions and OOF arrays to Drive
drive_dest = '/content/drive/MyDrive/kaggle_airline_output'
if os.path.exists('/content/drive/MyDrive'):
    os.makedirs(drive_dest, exist_ok=True)
    for f in glob.glob('submissions/*.csv') + glob.glob('oof/*.npy'):
        shutil.copy(f, drive_dest)
    print(f'[OK] Backed up all outputs to Google Drive: {drive_dest}')

# 2. Automatically trigger browser downloads for each generated submission CSV
try:
    from google.colab import files
    sub_files = glob.glob('submissions/*.csv')
    print(f'[*] Triggering browser download for {len(sub_files)} submission files:')
    for f in sorted(sub_files):
        print(f'  - Downloading: {f}')
        files.download(f)
    print('[OK] Download requests dispatched to browser.')
except ImportError:
    print('[*] Local environment detected; submissions saved in submissions/ directory.')
